In [1]:
import pandas as pd
# from main import SatCLIPLightningModule
# import location_encoder as LE
# from temporal_encoding import Fourier, Direct
from tqdm import tqdm
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import numpy as np

from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import io
from PIL import Image

from utils import *
import torch.nn.functional as F

In [2]:
# Load the GHCN dataset
ghcn_df = pd.read_csv('../data/ghcn_data/ghcn_2021_2026_temperature.csv')
print(len(ghcn_df))
ghcn_df.head()

104066815


,ID,Date,Element,Value,MFlag,QFlag,SFlag,OBS-TIME,Latitude,Longitude,Elevation,Name
0,AE000041196,20210101,TMAX,278,NaN,NaN,S,NaN,25.333,55.517,34.0,SHARJAH INTER. AIRP
1,AE000041196,20210101,PRCP,0,D,NaN,S,NaN,25.333,55.517,34.0,SHARJAH INTER. AIRP
2,AEM00041194,20210101,TMAX,266,NaN,NaN,S,NaN,25.255,55.364,10.4,DUBAI INTL
3,AEM00041194,20210101,TMIN,178,NaN,NaN,S,NaN,25.255,55.364,10.4,DUBAI INTL
4,AEM00041194,20210101,PRCP,0,NaN,NaN,S,NaN,25.255,55.364,10.4,DUBAI INTL


In [6]:
# Filter GHCN to quality controlled data and limit to TMAX (maximum temperature) observations
ghcn_qc_temp_df = ghcn_df[(ghcn_df["Element"]=="TMAX") & (ghcn_df["QFlag"].isna())].copy()
ghcn_qc_temp_df['Date'] = pd.to_datetime(ghcn_qc_temp_df['Date'], format="%Y%m%d")
ghcn_qc_temp_df.loc[:, 'DayOfYear'] = (ghcn_qc_temp_df['Date'].dt.dayofyear - 1) / 364.0
ghcn_qc_temp_df.loc[:, 'POSIX_TIME'] = ghcn_qc_temp_df['Date'].astype(np.int64) // 10**9
ghcn_qc_temp_df.head()

,ID,Date,Element,Value,MFlag,QFlag,SFlag,OBS-TIME,Latitude,Longitude,Elevation,Name,DayOfYear,POSIX_TIME
0,AE000041196,2021-01-01,TMAX,278,NaN,NaN,S,NaN,25.3330,55.5170,34.0,SHARJAH INTER. AIRP,0.0,1609459200
2,AEM00041194,2021-01-01,TMAX,266,NaN,NaN,S,NaN,25.2550,55.3640,10.4,DUBAI INTL,0.0,1609459200
5,AEM00041217,2021-01-01,TMAX,262,NaN,NaN,S,NaN,24.4330,54.6510,26.8,ABU DHABI INTL,0.0,1609459200
7,AEM00041218,2021-01-01,TMAX,250,NaN,NaN,S,NaN,24.2620,55.6090,264.9,AL AIN INTL,0.0,1609459200
67,ASN00031108,2021-01-01,TMAX,290,NaN,NaN,a,NaN,-17.1347,145.4281,594.0,WALKAMIN RESEARCH STATION,0.0,1609459200


In [ ]:
export_data = ghcn_qc_temp_df[['Latitude', 'Longitude', 'POSIX_TIME', 'Value']]
export_data.to_csv('../data/ghcn_2021_2026_high_temp_benchmark.csv', index=False, header=False)

Num_observations:  22614137
Num_stations:  14677


In [ ]:
print("Num_observations: ", len(export_data))
print("Num_stations: ", len(export_data.groupby(['Latitude', 'Longitude'])))

In [ ]:
# Plot the splits on a map
fig, ax = plt.subplots(figsize=(12, 6))
m = Basemap(projection='cyl', resolution='c', ax=ax)
m.drawcoastlines()
# Parallels (Latitudes) range from -90 to 90
m.drawparallels(np.arange(-90, 90, 30), labels=[True, True, False, False])

# Meridians (Longitudes) range from 0 to 360 (or -180 to 180 depending on projection)
m.drawmeridians(np.arange(0, 360, 30), labels=[False, False, True, True])

sc = ax.scatter(export_data["Latitude"], export_data["Longitude"], c=export_data["Value"]/10, cmap='coolwarm', marker='o', alpha=0.5)
fig.colorbar(sc, label='Split', ax=ax)